## Google Cloud Key Management

# Core Cryptographic Key Management via Google Cloud KMS

The **Google Cloud Key Management Service (KMS)** is an enterprise-grade, fully managed service that allows you to create, manage, use, and audit cryptographic keys to protect sensitive cloud infrastructure and application payloads.

Google Cloud KMS supports both symmetric and asymmetric key structures, enabling a wide range of cryptographic operations including hardware-backed encryption, decryption, algorithmic signing, and signature verification.

---

## Core Structural Concepts

Google Cloud KMS organises key material in a rigid hierarchical structure to simplify Access Control (IAM) and security auditing:

* **Key Rings:** Logical, structural containers used to organize related keys. IAM access policies applied to a Key Ring are automatically inherited by all keys nested inside it.
* **CryptoKeys:** The actual logical cryptographic keys stored within a Key Ring. A CryptoKey defines the purpose (e.g., symmetric encryption vs. asymmetric signing) and contains the version ledger.
* **CryptoKey Versions:** The actual underlying raw cryptographic key material. Each CryptoKey can possess multiple sequential versions to facilitate automated lifecycle rotations.
* **Symmetric Keys:** Algorithmic structures that use the exact same shared key material to handle both data encryption and data decryption operations.
* **Asymmetric Keys:** Cryptographic structures that split roles into a publicly distributable key (for data encryption or signature validation) and a heavily guarded private key (for data decryption or signature generation).

---

## Real-World Application: Secure File Storage & Envelope Encryption

To understand how Google Cloud KMS operates at scale, consider a secure file storage application. Direct data encryption via cloud APIs is restricted to small payloads (up to **64 KiB**). For larger file assets, applications must implement an industry pattern known as **Envelope Encryption**.

### The Envelope Encryption Process:

1. **Data Encryption Key (DEK) Generation:** When a user uploads a large file (e.g., a 5 GB backup video), the application generates a unique, single-use Symmetric Data Encryption Key (DEK) locally in memory.
2. **Data Ingest Encryption:** The local server encrypts the large file body using a fast, local block cipher (like AES-256) driven by the plaintext DEK.
3. **Key Wrapping:** The server transmits the tiny 256-bit plaintext DEK over a secure TLS connection to Google Cloud KMS. KMS encrypts (wraps) the DEK using a master **CryptoKey** managed safely inside Google's HSM infrastructure.
4. **Storage:** The application writes the heavily encrypted file data block alongside the wrapped (encrypted) DEK to a storage target like Cloud Storage. The plaintext DEK is immediately scrubbed from application memory.
5. **Secure Decryption (Unwrapping):** When retrieving the file, the application sends the encrypted DEK back to KMS. If the service account has IAM permissions, KMS decrypts (unwraps) the DEK and returns it. The application then decrypts the local file block in memory.

---

## Programmatic Implementation with Python

The following practical scripts demonstrate how to drive the lifecycle of cryptographic resources using the official `google-cloud-kms` SDK.

### 1. Provisioning a Key Ring and CryptoKey

Before generating keys, you must establish a localized Key Ring container.

```python
from google.cloud import kms

# Initialize the gRPC Key Management Service client
client = kms.KeyManagementServiceClient()

# Configure resource coordinates
project_id = "your-project-id"
location_id = "global"  # Can be regional (e.g., us-central1) or multi-regional
key_ring_id = "sample-key-ring"
crypto_key_id = "sample-crypto-key"

# Compile the parent path resource coordinate string
parent_path = f"projects/{project_id}/locations/{location_id}"

# Step A: Create the logical Key Ring container
key_ring = client.create_key_ring(
    request={
        "parent": parent_path,
        "key_ring_id": key_ring_id,
        "key_ring": {},
    }
)
print(f"Key Ring created successfully: {key_ring.name}")

# Step B: Create a symmetric encryption CryptoKey inside the new Key Ring
crypto_key = client.create_crypto_key(
    request={
        "parent": key_ring.name,
        "crypto_key_id": crypto_key_id,
        "crypto_key": {
            # Purpose specifies the intended algorithmic capability restrictions
            "purpose": kms.CryptoKey.CryptoKeyPurpose.ENCRYPT_DECRYPT,
        },
    }
)
print(f"CryptoKey initialized: {crypto_key.name}")

```

---

### 2. Auditing CryptoKey Metadata

You can inspect the structural parameters, active states, and rotational settings of an asset without interacting with or exposing the raw key material.

```python
# Construct the fully qualified resource path string
crypto_key_name = f"projects/{project_id}/locations/{location_id}/keyRings/{key_ring_id}/cryptoKeys/{crypto_key_id}"

# Retrieve metadata state payload
key_meta = client.get_crypto_key(request={"name": crypto_key_name})

print("--- Key Metadata Audit ---")
print(f"Key Resource Coordinates: {key_meta.name}")
print(f"Cryptographic Purpose: {key_meta.purpose.name}")
print(f"Primary Active Version: {key_meta.primary.name}")
print(f"Timestamp Created: {key_meta.create_time}")
print(f"Active Automated Rotation Period: {key_meta.rotation_period}")

```

**Console Output:**

```text
--- Key Metadata Audit ---
Key Resource Coordinates: projects/your-project-id/locations/global/keyRings/sample-key-ring/cryptoKeys/sample-crypto-key
Cryptographic Purpose: ENCRYPT_DECRYPT
Primary Active Version: projects/your-project-id/locations/global/keyRings/sample-key-ring/cryptoKeys/sample-crypto-key/cryptoKeyVersions/1
Timestamp Created: 2026-05-29 14:22:11.123456+00:00
Active Automated Rotation Period: None

```

---

### 3. Direct Inline Encryption and Decryption (Payloads < 64 KiB)

For small values like configuration strings, API tokens, or salts, you can invoke the KMS `encrypt` and `decrypt` API endpoints directly.

```python
# Raw text payload to protect (must be bytes format)
plaintext_data = b"Hello, Google Cloud!"

# Direct server-side encryption request
encrypt_response = client.encrypt(
    request={
        "name": crypto_key_name,
        "plaintext": plaintext_data,
    }
)
ciphertext_blob = encrypt_response.ciphertext
print(f"Data successfully encrypted. Ciphertext size: {len(ciphertext_blob)} bytes")

# Direct server-side decryption request
decrypt_response = client.decrypt(
    request={
        "name": crypto_key_name,
        "ciphertext": ciphertext_blob,
    }
)
recovered_text = decrypt_response.plaintext.decode("utf-8")
print(f"Decrypted text: {recovered_text}")

```

---

### 4. Implementing Envelope Encryption (Payloads > 64 KiB)

The following script demonstrates generating an isolated data key, wrapping it using Cloud KMS for storage, and unwrapping it for decryption.

```python
import os

# 1. Generate a cryptographically secure 256-bit AES Data Encryption Key (DEK) local in memory
local_dek = os.urandom(32)

# 2. Wrap (encrypt) the local DEK using the Cloud KMS CryptoKey
wrap_response = client.encrypt(
    request={
        "name": crypto_key_name,
        "plaintext": local_dek,
    }
)
encrypted_dek_blob = wrap_response.ciphertext
print("DEK wrapped successfully. Storable ciphertext generated.")

# --- Storage phase takes place: Save encrypted_dek_blob alongside data asset ---

# 3. Unwrap (decrypt) the storable DEK blob back into memory when file access is requested
unwrap_response = client.decrypt(
    request={
        "name": crypto_key_name,
        "ciphertext": encrypted_dek_blob,
    }
)
unwrapped_plaintext_dek = unwrap_response.plaintext

# Verify that memory bytes safely match our source key material
assert local_dek == unwrapped_plaintext_dek
print("DEK unwrapped successfully. Plaintext key restored to secure processing memory.")

```

---

## Administrative Key Lifecycle Management

### 1. Configuring Automatic Key Rotation

Key rotation limits the volume of data protected by a single key version, mitigating the blast radius if an individual version is compromised.

```python
import time
from google.protobuf import duration_pb2

# Define a 30-day rotation cadence constraint using Protobuf Durations
rotation_cadence = duration_pb2.Duration()
rotation_cadence.seconds = 60 * 60 * 24 * 30  # 30 Days converted to seconds

# Commit the configuration mask update
updated_key_spec = client.update_crypto_key(
    request={
        "crypto_key": {
            "name": crypto_key_name,
            "rotation_period": rotation_cadence,
            "next_rotation_time": {"seconds": int(time.time() + rotation_cadence.seconds)},
        },
        # Use field masks to safely update properties without resetting other parameters
        "update_mask": {"paths": ["rotation_period", "next_rotation_time"]},
    }
)
print("Automated rotation schedule applied successfully.")

```

> ⚠️ **Critical Architectural Note:** Enabling automated rotation generates a new **CryptoKey Version** for future encryption operations. It **does not** retroactively re-encrypt existing ciphertexts stored in your databases.
> Old ciphertexts remain safely bound to their historical key versions. Cloud KMS maintains all historical versions transparently, allowing your applications to seamlessly decrypt legacy files using older versions while using the new primary version for all incoming write operations.

### 2. Scheduling Key Destruction

When decommissioning legacy systems or rotating keys out of service completely, you can schedule the destruction of explicit key versions.

```python
# Target the explicit primary version path string
target_version_to_purge = key_meta.primary.name

# Issue destruction request
destroy_execution = client.destroy_crypto_key_version(
    request={"name": target_version_to_purge}
)
print(f"Administrative Action: Version '{target_version_to_purge}' scheduled for destruction.")

```

**Console Output:**

```text
Administrative Action: Version 'projects/your-project-id/locations/global/keyRings/sample-key-ring/cryptoKeys/sample-crypto-key/cryptoKeyVersions/1' scheduled for destruction.

```

> 🛑 **The Safety Buffer Window:** Destroying a key version enters a scheduled hold state (typically **24 hours** by default). During this buffer period, the key cannot be used for cryptographic operations, but the destruction request can be aborted if an emergency system dependency is discovered. Once this window closes, the key material is destroyed within the physical HSM modules and data recovery becomes completely impossible.

---

## Summary

In this lesson, we explored the structural concepts and administrative capabilities of Google Cloud Key Management Service (KMS):

* **Hierarchy Enforcement:** Grouping logical `CryptoKeys` inside geographical `Key Rings` to simplify security permissions.
* **Envelope Encryption Blueprinting:** Circumventing the 64 KiB direct API transport limit by generating unique local DEKs and wrapping them securely via KMS master keys.
* **Rotational Automation:** Updating key parameters dynamically using `FieldMask` pipelines while preserving historical versions to maintain seamless readability.
* **Lifecycle Decommissioning:** Scheduling permanent key version destruction with built-in safety buffer windows.

## Creating Google Cloud KMS Key with Python SDK

As a precursor, you've acquired a solid understanding of how to manage keys in Google Cloud KMS using Google Cloud Python SDK. In this task, you will leverage this knowledge and put it into practice. Your mission is to run a provided Python script that creates a Google Cloud KMS key using Google Cloud Key Management Service (KMS). While running the script, observe the output for a better understanding of how the key is created.

Important Note: Running scripts can modify the resources in our Google Cloud simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
import json
from unittest.mock import MagicMock
from google.cloud import kms

class MockKeyManagementServiceClient:
    def __init__(self):
        self.key_rings = {}
        self.crypto_keys = {}
        self.key_versions = {}
        
    def create_key_ring(self, request):
        parent = request['parent']
        key_ring_id = request['key_ring_id']
        key_ring_name = f"{parent}/keyRings/{key_ring_id}"
        
        mock_key_ring = MagicMock()
        mock_key_ring.name = key_ring_name
        self.key_rings[key_ring_name] = mock_key_ring
        return mock_key_ring
        
    def create_crypto_key(self, request):
        parent = request['parent']
        crypto_key_id = request['crypto_key_id']
        crypto_key = request['crypto_key']
        crypto_key_name = f"{parent}/cryptoKeys/{crypto_key_id}"
        
        mock_crypto_key = MagicMock()
        mock_crypto_key.name = crypto_key_name
        mock_crypto_key.purpose = crypto_key.get('purpose', kms.CryptoKey.CryptoKeyPurpose.ENCRYPT_DECRYPT)
        mock_crypto_key.create_time = MagicMock()
        mock_crypto_key.labels = {}
        mock_crypto_key.rotation_period = None
        
        # Create a primary version
        version_name = f"{crypto_key_name}/cryptoKeyVersions/1"
        mock_version = MagicMock()
        mock_version.name = version_name
        mock_version.state = kms.CryptoKeyVersion.CryptoKeyVersionState.ENABLED
        mock_crypto_key.primary = mock_version
        
        self.crypto_keys[crypto_key_name] = mock_crypto_key
        self.key_versions[crypto_key_name] = [mock_version]
        return mock_crypto_key
        
    def get_crypto_key(self, request):
        name = request['name']
        return self.crypto_keys.get(name, self._create_default_crypto_key(name))
        
    def encrypt(self, request):
        name = request['name']
        plaintext = request['plaintext']
        
        # Simple mock encryption (just base64 encode for demo)
        import base64
        ciphertext = base64.b64encode(plaintext).decode('utf-8')
        
        mock_response = MagicMock()
        mock_response.ciphertext = ciphertext.encode('utf-8')
        mock_response.name = name
        return mock_response
        
    def decrypt(self, request):
        name = request['name']
        ciphertext = request['ciphertext']
        
        # Simple mock decryption (base64 decode)
        import base64
        try:
            if isinstance(ciphertext, bytes):
                ciphertext = ciphertext.decode('utf-8')
            plaintext = base64.b64decode(ciphertext)
        except:
            plaintext = b"decrypted text"
            
        mock_response = MagicMock()
        mock_response.plaintext = plaintext
        mock_response.name = name
        return mock_response
        
    def update_crypto_key(self, request):
        crypto_key = request['crypto_key']
        name = crypto_key['name']
        
        if name in self.crypto_keys:
            existing_key = self.crypto_keys[name]
            if 'rotation_period' in crypto_key:
                existing_key.rotation_period = crypto_key['rotation_period']
            if 'next_rotation_time' in crypto_key:
                existing_key.next_rotation_time = crypto_key['next_rotation_time']
            return existing_key
        else:
            return self._create_default_crypto_key(name)
            
    def destroy_crypto_key_version(self, request):
        name = request['name']
        
        mock_response = MagicMock()
        mock_response.name = name
        mock_response.destroy_time = MagicMock()
        mock_response.state = kms.CryptoKeyVersion.CryptoKeyVersionState.DESTROY_SCHEDULED
        return mock_response
        
    def list_crypto_keys(self, request):
        parent = request['parent']
        return [key for name, key in self.crypto_keys.items() if name.startswith(f"{parent}/cryptoKeys/")]
        
    def list_key_rings(self, request):
        parent = request['parent']
        return [ring for name, ring in self.key_rings.items() if name.startswith(f"{parent}/keyRings/")]
        
    def _create_default_crypto_key(self, name):
        mock_crypto_key = MagicMock()
        mock_crypto_key.name = name
        mock_crypto_key.purpose = kms.CryptoKey.CryptoKeyPurpose.ENCRYPT_DECRYPT
        mock_crypto_key.create_time = MagicMock()
        mock_crypto_key.labels = {}
        mock_crypto_key.rotation_period = None
        
        # Create a primary version
        version_name = f"{name}/cryptoKeyVersions/1"
        mock_version = MagicMock()
        mock_version.name = version_name
        mock_version.state = kms.CryptoKeyVersion.CryptoKeyVersionState.ENABLED
        mock_crypto_key.primary = mock_version
        
        return mock_crypto_key


# Monkey patch the client
import sys
import google.cloud.kms
google.cloud.kms.KeyManagementServiceClient = MockKeyManagementServiceClient

import json
from unittest.mock import MagicMock

class MockSecretManagerServiceClient:
    def __init__(self):
        self.secrets = {}
        self.versions = {}
        
    def create_secret(self, request):
        secret_id = request['secret_id']
        parent = request['parent']
        secret_name = f"{parent}/secrets/{secret_id}"
        
        mock_secret = MagicMock()
        mock_secret.name = secret_name
        self.secrets[secret_name] = mock_secret
        self.versions[secret_name] = []
        return mock_secret
        
    def add_secret_version(self, request):
        parent = request['parent']
        payload_data = request['payload']['data']
        
        version_num = len(self.versions[parent]) + 1
        version_name = f"{parent}/versions/{version_num}"
        
        mock_version = MagicMock()
        mock_version.name = version_name
        mock_version.payload = MagicMock()
        mock_version.payload.data = payload_data
        
        self.versions[parent].append(mock_version)
        return mock_version
        
    def access_secret_version(self, request):
        name = request['name']
        if '/versions/latest' in name:
            secret_name = name.replace('/versions/latest', '')
            if secret_name in self.versions and self.versions[secret_name]:
                return self.versions[secret_name][-1]
        
        # Handle specific version access
        if '/versions/' in name:
            parts = name.split('/versions/')
            if len(parts) == 2:
                secret_name = parts[0]
                version_num = parts[1]
                if secret_name in self.versions:
                    try:
                        version_index = int(version_num) - 1
                        if 0 <= version_index < len(self.versions[secret_name]):
                            return self.versions[secret_name][version_index]
                    except ValueError:
                        pass
        
        mock_response = MagicMock()
        mock_response.payload = MagicMock()
        mock_response.payload.data = b'{"username": "test", "password": "password"}'
        return mock_response
        
    def list_secret_versions(self, request):
        parent = request['parent']
        return self.versions.get(parent, [])
        
    def update_secret(self, secret=None, update_mask=None, request=None):
        # Support both direct arguments and request dict for compatibility
        if request is not None:
            secret_info = request['secret']
        else:
            secret_info = secret
        secret_name = secret_info['name']

        mock_secret = MagicMock()
        mock_secret.name = secret_name

        # Handle labels update
        if 'labels' in secret_info:
            mock_secret.labels = secret_info['labels']

        return mock_secret
    
    def delete_secret(self, request=None, name=None):
        # Handle both calling patterns
        if request is not None:
            secret_name = request['name']
        elif name is not None:
            secret_name = name
        else:
            raise ValueError("Either 'request' or 'name' parameter must be provided")
        
        if secret_name in self.secrets:
            del self.secrets[secret_name]
            if secret_name in self.versions:
                del self.versions[secret_name]
        
        mock_response = MagicMock()
        return mock_response

    def get_secret(self, request):
        name = request['name']
        
        mock_secret = MagicMock()
        mock_secret.name = name
        mock_secret.replication = {"automatic": {}}
        mock_secret.create_time = MagicMock()
        mock_secret.labels = {}
        
        return mock_secret
    
    def list_secrets(self, request):
        parent = request['parent']
        # Return all secrets under the given parent
        return [secret for name, secret in self.secrets.items() if name.startswith(f"{parent}/secrets/")]
        
    def destroy_secret_version(self, request):
        name = request['name']
        # name format: .../secrets/{secret_id}/versions/{version_id}
        parts = name.split('/secrets/')
        if len(parts) == 2:
            secret_and_version = parts[1]
            secret_id, _, version_id = secret_and_version.partition('/versions/')
            secret_name = f"{parts[0]}/secrets/{secret_id}"
            if secret_name in self.versions:
                version_index = int(version_id) - 1
                if 0 <= version_index < len(self.versions[secret_name]):
                    del self.versions[secret_name][version_index]
        mock_response = MagicMock()
        return mock_response


# Monkey patch the client
import sys
import google.cloud.secretmanager
google.cloud.secretmanager.SecretManagerServiceClient = MockSecretManagerServiceClient

import mock_kms
from google.cloud import kms

# Initialize KMS client
client = kms.KeyManagementServiceClient()

# Define project, location, key ring, and crypto key
project_id = 'sample-project-id'  
location_id = 'global'
key_ring_id = 'sample-key-ring'
crypto_key_id = 'sample-crypto-key'

# Create the parent path for key ring
parent = f"projects/{project_id}/locations/{location_id}"

# Create key ring first (required for Google Cloud KMS)
try:
    key_ring_response = client.create_key_ring(
        request={
            "parent": parent,
            "key_ring_id": key_ring_id,
        }
    )
except Exception:
    # Key ring might already exist, continue
    pass

# Create crypto key within the key ring
key_ring_name = f"{parent}/keyRings/{key_ring_id}"
crypto_key = {
    "purpose": kms.CryptoKey.CryptoKeyPurpose.ENCRYPT_DECRYPT,
}

crypto_key_response = client.create_crypto_key(
    request={
        "parent": key_ring_name,
        "crypto_key_id": crypto_key_id,
        "crypto_key": crypto_key
    }
)

# Extract Key name (equivalent to Key ID in AWS)
key_name = crypto_key_response.name
print('Created key with Key name: ', key_name)
```

## Google Cloud KMS Data Key Generation with Envelope Encryption

Skip to main content
Google Cloud Key Management

We're now moving forward! In this task, you'll extend the functionality of a Python script for generating a data key in Google Cloud KMS. The provided script initiates a Google Cloud KMS client, creates a crypto key, and extracts the key_name from the KMS key response. Your task is to complete the script by adding functionality to generate a data key using envelope encryption pattern - generating a data encryption key (DEK) locally and encrypting it with the KMS crypto key. Run your updated script and observe the response.

Important Note: Running scripts can modify the resources in our Google Cloud simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to your clipboard.


```python
import mock_kms
import os
from google.cloud import kms

# Initialize KMS client
client = kms.KeyManagementServiceClient()

# Define project, location, key ring, and crypto key
project_id = 'sample-project-id'  
location_id = 'global'
key_ring_id = 'sample-key-ring'
crypto_key_id = 'sample-crypto-key'

# Create the parent path for key ring
parent = f"projects/{project_id}/locations/{location_id}"

# Create key ring first (required for Google Cloud KMS)
try:
    key_ring_response = client.create_key_ring(
        request={
            "parent": parent,
            "key_ring_id": key_ring_id,
        }
    )
except Exception:
    # Key ring might already exist, continue
    pass

# Create crypto key within the key ring
key_ring_name = f"{parent}/keyRings/{key_ring_id}"
crypto_key = {
    "purpose": kms.CryptoKey.CryptoKeyPurpose.ENCRYPT_DECRYPT,
}

crypto_key_response = client.create_crypto_key(
    request={
        "parent": key_ring_name,
        "crypto_key_id": crypto_key_id,
        "crypto_key": crypto_key
    }
)

# Extract Key name (equivalent to Key ID in AWS)
key_name = crypto_key_response.name
print('Created key with Key name: ', key_name)

# TODO: Generate a data key using envelope encryption
```

Here is the complete script implementing the **Envelope Encryption** pattern. It uses Python's secure `os.urandom()` module to generate a fresh 256-bit Data Encryption Key (DEK) locally in memory, then uses the newly created KMS master key to encrypt (wrap) it for secure storage alongside your data.

```python
import mock_kms
import os
from google.cloud import kms

# Initialize KMS client
client = kms.KeyManagementServiceClient()

# Define project, location, key ring, and crypto key
project_id = 'sample-project-id'  
location_id = 'global'
key_ring_id = 'sample-key-ring'
crypto_key_id = 'sample-crypto-key'

# Create the parent path for key ring
parent = f"projects/{project_id}/locations/{location_id}"

# Create key ring first (required for Google Cloud KMS)
try:
    key_ring_response = client.create_key_ring(
        request={
            "parent": parent,
            "key_ring_id": key_ring_id,
        }
    )
except Exception:
    # Key ring might already exist, continue
    pass

# Create crypto key within the key ring
key_ring_name = f"{parent}/keyRings/{key_ring_id}"
crypto_key = {
    "purpose": kms.CryptoKey.CryptoKeyPurpose.ENCRYPT_DECRYPT,
}

crypto_key_response = client.create_crypto_key(
    request={
        "parent": key_ring_name,
        "crypto_key_id": crypto_key_id,
        "crypto_key": crypto_key
    }
)

# Extract Key name (equivalent to Key ID in AWS)
key_name = crypto_key_response.name
print('Created key with Key name: ', key_name)

# ========================================================
# TODO FIXED: GENERATE A DATA KEY USING ENVELOPE ENCRYPTION
# ========================================================

# 1. Generate a cryptographically secure 256-bit (32-byte) DEK locally in application memory
plaintext_dek = os.urandom(32)
print(f"\nGenerated Local DEK in memory ({len(plaintext_dek) * 8} bits)")

# 2. Encrypt (wrap) the plaintext DEK using the Cloud KMS master key
encrypt_response = client.encrypt(
    request={
        "name": key_name,
        "plaintext": plaintext_dek,
    }
)

# 3. Extract the encrypted DEK ciphertext blob
encrypted_dek = encrypt_response.ciphertext

print("--- Envelope Encryption Artifacts ---")
print(f"Plaintext DEK (Hex representation): {plaintext_dek.hex()}")
print(f"Encrypted DEK Length: {len(encrypted_dek)} bytes")
print("Status: Success! The encrypted DEK can now be safely stored next to your encrypted file.")

```

---

### Architectural Flow of Envelope Encryption

Implementing this code showcases a crucial security architectural blueprint:

* **The Local Cipher:** Your large production datasets are encrypted locally inside your app using the fast, local `plaintext_dek` byte stream via block ciphers like AES-256.
* **The Wrapped Shield:** You never write the plaintext DEK to persistent disks. Instead, you only store the `encrypted_dek` blob. Because it is securely wrapped by the KMS master key, an attacker who compromises your database layer cannot read your underlying files without gaining explicit IAM access permissions to your Cloud KMS `key_name`.

## Fetch Google Cloud KMS Key Metadata and Display State

## Enable Key Rotation with Google Cloud KMS

## Google Cloud KMS Complete Lifecycle Implementation